# Qualitative Comparative Analysis (fsQCA)

Este notebook comienza **después de la calibración** realizada en `preQCA`.

**Entrada:** `qca_calibrada.csv`

Cada caso representa un **gobierno regional en un año determinado (GORE–año)**. Analizaremos las condiciones `ALCANCE`, `COMPARTIDAS`, `CONCENTRACION` e `INTERMEDIACION`, y el resultado `INCO_ALTO`.

## ¿Qué haremos y para qué?

1. **Leer la base calibrada** para verificar los casos y las membresías que entran al análisis.
2. **Evaluar necesidad** para examinar si alguna condición, o su ausencia, es necesaria para `INCO_ALTO`.
3. **Construir la tabla de verdad** para organizar las combinaciones de condiciones y reconocer cuáles están representadas por los casos.
4. **Identificar configuraciones suficientes** para examinar qué combinaciones se asocian sistemáticamente con `INCO_ALTO`.
5. **Simplificar las configuraciones** mediante minimización lógica para obtener una expresión más parsimoniosa.
6. **Evaluar la solución** mediante consistencia, cobertura y PRI.

En cada paso ejecutaremos un bloque pequeño, observaremos los resultados e interpretaremos su significado para las redes de contratación de los gobiernos regionales. Los hallazgos estarán sujetos a los casos incluidos y a los umbrales provisionales de calibración utilizados.

Usaremos `setqca`, una implementación de fsQCA para Python.

In [1]:
!pip -q install setqca

## QCA-01: leer la base preparada

Cada fila representa un gobierno regional en un año determinado (GORE–año).

`ALCANCE`, `COMPARTIDAS`, `CONCENTRACION` e `INTERMEDIACION` son las membresías fuzzy de las condiciones, mientras que `INCO_ALTO` es la membresía del resultado.

Todas toman valores entre 0 y 1 y fueron construidas mediante la calibración realizada en `preQCA`.

In [2]:
import pandas as pd

qca = pd.read_csv(
    "https://raw.githubusercontent.com/romloayza/networks_x_qca_tesis/refs/heads/main/data/qca_calibrada_tesis.csv"
)

qca.head()

,caso,ALCANCE,COMPARTIDAS,CONCENTRACION,INTERMEDIACION,INCO_ALTO
0,AMAZONAS_2021,0.859599,0.197481,0.101024,0.939095,0.936681
1,AMAZONAS_2022,0.516764,0.039148,0.126381,0.267202,0.926370
2,AMAZONAS_2024,0.680313,0.151328,0.181488,0.491741,0.463261
3,ANCASH_2021,0.999779,0.042173,0.037640,0.999646,0.996754
4,ANCASH_2022,0.999974,0.063233,0.028406,0.999984,0.993441


## QCA-02: buscar condiciones necesarias

Una condición necesaria responde a la pregunta:

> Cuando encontramos un GORE–año con pertenencia a `INCO_ALTO`, ¿encontramos también pertenencia a la condición?

Usaremos `necessity_analysis()` para evaluar `ALCANCE`, `COMPARTIDAS`, `CONCENTRACION` e `INTERMEDIACION`, así como sus ausencias, respecto de `INCO_ALTO`.

Tomamos una consistencia de 0.90 como referencia para identificar una relación de necesidad fuerte. También examinaremos la cobertura para valorar la relevancia empírica de esa relación.

In [3]:
from setqca import necessity_analysis

necesidad = necessity_analysis(
    qca,
    outcome="INCO_ALTO",
    conditions=["ALCANCE", "COMPARTIDAS", "CONCENTRACION", "INTERMEDIACION"],
    consistency_threshold=0.90
)

necesidad.to_frame()[
    ["condition", "consistency", "coverage", "necessary"]
]

,condition,consistency,coverage,necessary
0,~COMPARTIDAS,0.723231,0.968580,False
1,~CONCENTRACION,0.699741,0.957049,False
2,~INTERMEDIACION,0.664059,0.962124,False
3,~ALCANCE,0.590525,0.961327,False
4,ALCANCE,0.477182,0.966792,False
5,INTERMEDIACION,0.411542,0.985369,False
6,CONCENTRACION,0.362724,0.962876,False
7,COMPARTIDAS,0.347352,0.961767,False


### ¿Cómo leer el resultado?

La columna principal para evaluar necesidad es `consistency`. Un valor cercano a 1 indica que los casos pertenecientes a `INCO_ALTO` también pertenecen sistemáticamente a la condición evaluada.

Ninguna de las cuatro condiciones, ni sus ausencias, alcanza el umbral de consistencia de 0.90. La mayor consistencia corresponde a `~COMPARTIDAS` (0.723), seguida de `~CONCENTRACION` (0.700). El símbolo `~` representa la negación del conjunto: por ejemplo, `~COMPARTIDAS` significa ausencia de pertenencia al conjunto de GORE–año con alta proporción de contratistas compartidos.

Por tanto, **ninguna condición individual ni su ausencia puede considerarse necesaria para `INCO_ALTO` bajo los criterios y la calibración utilizados**.

Aunque las coberturas son elevadas, esto no compensa las consistencias inferiores al umbral de necesidad.

Este resultado no implica que las condiciones carezcan de importancia. En los siguientes pasos examinaremos si sus **combinaciones** constituyen configuraciones suficientes para `INCO_ALTO`.

## QCA-03: observar combinaciones

Con cuatro condiciones existen `2⁴ = 16` configuraciones lógicamente posibles de `ALCANCE`, `COMPARTIDAS`, `CONCENTRACION` e `INTERMEDIACION`.

`build_truth_table()` organiza esas configuraciones. Los casos GORE–año se ubican en una fila según estén más dentro —membresía superior a 0.5— o más fuera —membresía inferior a 0.5— de cada conjunto. Los casos con membresía exactamente igual a 0.5 requieren revisión, pues se encuentran en el punto de máxima ambigüedad.

La consistencia de cada fila respecto de `INCO_ALTO` se calcula utilizando las membresías fuzzy originales. Esta organización no convierte el análisis en QCA de conjuntos nítidos.

La tabla permitirá identificar qué configuraciones tienen casos observados y cuáles carecen de ellos.

In [6]:
qca_tabla = qca.loc[
    ~qca["caso"].isin(casos_cruce["caso"])
].copy()

tabla = build_truth_table(
    qca_tabla,
    outcome="INCO_ALTO",
    conditions=["ALCANCE", "COMPARTIDAS", "CONCENTRACION", "INTERMEDIACION"],
    inclusion_cutoff=0.80,
    pri_cutoff=0.75,
    frequency_cutoff=1,
    case_id="caso"
)

tabla_df = tabla.to_frame()

tabla_df

,ALCANCE,COMPARTIDAS,CONCENTRACION,INTERMEDIACION,minterm,n,consistency,PRI,OUT,cases,excluded_because
0,0,0,0,0,0,11,0.997653,0.996922,1,"CAJAMARCA_2022, CALLAO_2022, CALLAO_2024, HUAN...",
1,0,0,0,1,1,0,1.000000,1.000000,R,,frequency 0 below the cutoff of 1
2,0,0,1,0,2,9,0.993668,0.991519,1,"APURIMAC_2022, AREQUIPA_2022, CUSCO_2021, CUSC...",
3,0,0,1,1,3,0,1.000000,1.000000,R,,frequency 0 below the cutoff of 1
4,0,1,0,0,4,4,1.000000,1.000000,1,"AYACUCHO_2022, CAJAMARCA_2024, TACNA_2024, UCA...",
5,0,1,0,1,5,1,1.000000,1.000000,1,JUNIN_2022,
6,0,1,1,0,6,9,0.959135,0.939661,1,"ANCASH_2024, APURIMAC_2021, AREQUIPA_2021, AYA...",
7,0,1,1,1,7,0,1.000000,1.000000,R,,frequency 0 below the cutoff of 1
8,1,0,0,0,8,12,0.996620,0.995222,1,"AMAZONAS_2022, AMAZONAS_2024, CALLAO_2021, HUA...",
9,1,0,0,1,9,17,0.994346,0.992760,1,"AMAZONAS_2021, ANCASH_2021, ANCASH_2022, HUANC...",


## QCA-04: identificar configuraciones suficientes

Ahora preguntamos:

> Cuando un GORE–año presenta una configuración de condiciones, ¿presenta también pertenencia a `INCO_ALTO` de manera sistemática?

En nuestra tabla, `OUT = 1` identifica las configuraciones que cumplen los criterios establecidos: consistencia mínima de 0.80, PRI mínimo de 0.75 y frecuencia mínima de un caso.

El análisis utiliza `qca_tabla`, que excluye provisionalmente los dos casos con membresía exactamente igual a 0.5 en `COMPARTIDAS`.

Mostramos solamente las columnas necesarias para interpretar la clasificación de las configuraciones como suficientes.

In [7]:
columnas = [
    "ALCANCE", "COMPARTIDAS", "CONCENTRACION", "INTERMEDIACION",
    "n", "consistency", "OUT"
]

suficientes = tabla_df[
    tabla_df["OUT"].astype(str) == "1"
][columnas]

suficientes

,ALCANCE,COMPARTIDAS,CONCENTRACION,INTERMEDIACION,n,consistency,OUT
0,0,0,0,0,11,0.997653,1
2,0,0,1,0,9,0.993668,1
4,0,1,0,0,4,1.000000,1
5,0,1,0,1,1,1.000000,1
6,0,1,1,0,9,0.959135,1
8,1,0,0,0,12,0.996620,1
9,1,0,0,1,17,0.994346,1
13,1,1,0,1,4,0.996817,1


### **¿Qué observamos?**
La tabla filtrada permite identificar qué configuraciones alcanzan el criterio de suficiencia.

Con solo tres condiciones sería posible inspeccionar estas filas a simple vista, pero esa inspección manual deja de ser práctica apenas crece el número de condiciones. Por eso no la hacemos aquí: en QCA-05 usamos minimización lógica para que Python encuentre la simplificación de forma sistemática.

## QCA-05: simplificar las configuraciones

Utilizamos `FSQCA` para minimizar las configuraciones suficientes asociadas con `INCO_ALTO` y obtener una expresión lógica más parsimoniosa.

La evaluación de la solución conserva las membresías fuzzy. Trabajaremos con `qca_tabla` para mantener los mismos casos incluidos en la tabla de verdad.

Aplicaremos los mismos criterios establecidos en QCA-03:

- consistencia mínima = 0.80;
- frecuencia mínima = 1;
- PRI mínimo = 0.75.

La minimización permitirá examinar si las configuraciones suficientes pueden resumirse en una o varias rutas que combinen `ALCANCE`, `COMPARTIDAS`, `CONCENTRACION` e `INTERMEDIACION`, o sus ausencias.

In [8]:
from setqca import FSQCA

modelo_qca = FSQCA(
    consistency=0.80,
    pri=0.75,
    frequency=1
)

resultado_qca = modelo_qca.fit(
    qca_tabla,
    outcome="INCO_ALTO",
    conditions=["ALCANCE", "COMPARTIDAS", "CONCENTRACION", "INTERMEDIACION"],
    case_id="caso"
)

In [9]:
resultado_qca.summary_frame("conservative")

,solution,consistency,coverage,PRI,n_implicants,n_literals
0,~ALCANCE*~INTERMEDIACION + ALCANCE*~COMPARTIDA...,0.965593,0.808039,0.960638,3,8


### ¿Qué pasó?

El análisis quedó almacenado en `resultado_qca`. La solución conservadora obtenida contiene **tres rutas**, con ocho literales en total; cada literal es una condición presente o negada dentro de una ruta.

### ¿Cómo leer el resultado?

Los símbolos de la expresión lógica se interpretan así:

- `*` significa **Y**: las condiciones deben combinarse.
- `+` significa **O**: representa rutas alternativas.
- `~` representa la **negación del conjunto**, cuya membresía es uno menos la membresía original.

Por ejemplo, `~ALCANCE*~INTERMEDIACION` combina la ausencia de alto alcance empresarial con la ausencia de alta intermediación.

La solución presenta una consistencia de **0.966**, una cobertura de **0.808** y un PRI de **0.961**. La cobertura indica cuánto de la membresía en `INCO_ALTO` queda cubierto por la solución; no es un porcentaje de varianza explicada.


## QCA-06: evaluar la solución

Finalmente examinaremos los parámetros de ajuste de la solución asociada con `INCO_ALTO`.

La **consistencia** indica en qué medida la membresía en la solución es un subconjunto de la membresía en `INCO_ALTO`: hasta qué punto las rutas identificadas se comportan como suficientes para el resultado.

La **cobertura** indica cuánto de la membresía en `INCO_ALTO` queda cubierto por la solución. No representa varianza explicada.

El **PRI** permite evaluar la ambigüedad de la relación de suficiencia: valores elevados indican menor solapamiento de esa relación con el resultado y con su ausencia.

La interpretación estará referida a los casos de `qca_tabla` y a los umbrales de calibración utilizados.

In [11]:
resumen = resultado_qca.summary_frame("conservative")

with pd.option_context("display.max_colwidth", None):
    display(resumen)

,solution,consistency,coverage,PRI,n_implicants,n_literals
0,~ALCANCE*~INTERMEDIACION + ALCANCE*~COMPARTIDAS*~CONCENTRACION + COMPARTIDAS*~CONCENTRACION*INTERMEDIACION,0.965593,0.808039,0.960638,3,8


## Interpretación de la solución conservadora

La solución conservadora identifica tres configuraciones asociadas con el resultado `INCO_ALTO`:

`~ALCANCE*~INTERMEDIACION + ALCANCE*~COMPARTIDAS*~CONCENTRACION + COMPARTIDAS*~CONCENTRACION*INTERMEDIACION`

En esta expresión, `*` significa combinación de condiciones —Y—, `+` representa rutas alternativas —O— y `~` indica la negación de un conjunto. En conjuntos fuzzy, esta negación se calcula como uno menos la membresía original. Por ello, “ausencia de alta concentración” no equivale necesariamente a “concentración muy baja”.

### Las tres rutas identificadas

| Ruta | Configuración | Interpretación |
|---|---|---|
| 1 | `~ALCANCE*~INTERMEDIACION` | Combina ausencia de alto alcance empresarial y ausencia de alta intermediación. |
| 2 | `ALCANCE*~COMPARTIDAS*~CONCENTRACION` | Combina alto alcance empresarial, ausencia de una alta proporción de contratistas compartidos y ausencia de alta concentración contractual. |
| 3 | `COMPARTIDAS*~CONCENTRACION*INTERMEDIACION` | Combina una alta proporción de contratistas compartidos, ausencia de alta concentración contractual y alta intermediación. |

Las condiciones que no aparecen en una ruta no son exigidas por su expresión simplificada. Por ejemplo, la primera ruta no establece un requisito específico sobre `COMPARTIDAS` o `CONCENTRACION`. Esto no demuestra que dichas condiciones sean irrelevantes en general.

Las rutas son alternativas y pueden solaparse: un mismo GORE–año puede pertenecer en distinto grado a más de una. Su presencia expresa equifinalidad, es decir, la identificación de diferentes configuraciones asociadas con un mismo resultado.

### Parámetros de ajuste

Los siguientes valores corresponden a la solución conjunta, no a cada ruta por separado:

| Parámetro | Valor | Interpretación |
|---|---:|---|
| Consistencia | 0.965593 | Indica en qué medida la membresía en la solución está contenida en la membresía en `INCO_ALTO`. |
| Cobertura | 0.808039 | Indica cuánto de la membresía en `INCO_ALTO` queda cubierto por la solución. |
| PRI | 0.960638 | Indica poca ambigüedad simultánea de la relación de suficiencia respecto del resultado y de su ausencia. |

La solución presenta una consistencia elevada y cubre aproximadamente el 80.8 % de la membresía en el resultado entre los casos analizados. Estos indicadores no representan precisión predictiva ni proporción de varianza explicada.

### Alcance de los hallazgos

Bajo la calibración y los criterios utilizados, la solución identifica tres rutas que se comportan conjuntamente como suficientes para `INCO_ALTO`. La alta concentración contractual no aparece como requisito explícito de ninguna ruta: su ausencia forma parte de dos configuraciones y la primera no exige una pertenencia determinada en esa condición. Esto no permite concluir que la concentración reduzca el riesgo o que la diversificación lo produzca.

Antes de formular conclusiones sustantivas más firmes, debemos examinar la distribución de `INCO_ALTO`, los casos asociados con cada ruta y sus parámetros individuales. Si la pertenencia al resultado es elevada en casi todos los casos, una consistencia alta puede ofrecer una capacidad limitada para distinguir entre configuraciones.